# Estabilidad térmica de BGDY: nucleación y propagación de roturas B–C

**Problema.** Al calentar la lámina se rompen enlaces B–C. Hay dos casos distintos:
- **enlace libre**: el B y el extremo de la cadena se separan y al menos uno no encuentra otro compañero; queda un C con un solo vecino o un B con dos. Es el defecto que interesa;
- **reorganización**: el B–C se rompe pero, a la vez, sus átomos forman otro enlace (B con otro C, o C–C entre cadenas vecinas). Los dos conservan su número de vecinos y la red cambia de topología sin dejar enlaces colgantes.

El cuaderno detecta cada rotura, la clasifica y calcula tasas por enlace para cada tipo.

Ni el tiempo hasta la primera rotura ni el Lindemann medio son buenos parámetros de estabilidad:
- el tiempo hasta la primera rotura **depende del tamaño**: con más enlaces, antes se rompe alguno;
- el Lindemann por pares acumulado desde t = 0 crece sin límite en cuanto un par se separa, mezcla la vibración térmica con los defectos y su umbral (0.06) es arbitrario.

**Parámetros intensivos que calcula este cuaderno**
1. **Tasa de rotura por enlace**, $k(T)$ en ps⁻¹ por enlace B–C: roturas por tiempo-enlace en riesgo,
$$k = \frac{n_\text{roturas}}{\sum_t N_\text{intactos}(t)\,\Delta t}.$$
Si cada enlace intacto se rompe como un proceso de Poisson independiente, es el estimador de máxima verosimilitud y no depende del tamaño de la celda. Se calcula para todas las roturas y por tipo ($k = k_\text{libre} + k_\text{reorg}$). Con varias T: $\ln k = \ln A - E_a/(k_B T)$.
2. **Propagación**: probabilidad de que una rotura vaya seguida de otras cerca, y **exponente de Avrami** $n$ de la fracción de enlaces rotos, $X(t) = 1 - \exp[-(t/\tau)^n]$. Con $n \approx 1$ las roturas son independientes (defectos locales, $\tau = 1/k$). Con $n > 1$ hay nucleación y crecimiento del daño; en 2D, nucleación continua y crecimiento lineal dan $n \approx 3$.
3. **Comprobación de tamaño**: todas las tablas y figuras separan por tamaño de celda (3x3, 10x10…). Si $k$ y $n$ coinciden entre tamaños, son propiedades de la lámina y no de la celda.

**Entradas**: trayectorias `.xyz` en formato extxyz (la línea de comentario de cada frame lleva `Lattice="…"` y `pbc="T T F"`, como las que escribe ASE). Un archivo por trayectoria. También lee `.traj` y cualquier otro formato de ASE. La temperatura se lee del nombre (`md_bgdy3x3_1750_long.xyz` → 1750 K). **El tiempo entre frames no está en el archivo**: hay que darlo en `DT_FRAME_FS`. Con `DEMO = True` genera trayectorias sintéticas con roturas y cascadas de parámetros conocidos y ejecuta todo el cuaderno.

## 1. Parámetros

In [ ]:
from pathlib import Path

# --- Trayectorias
CARPETA_XYZ = Path(".")                    # se buscan los archivos aquí y en todas las subcarpetas
PATRON_XYZ = "*.xyz"
ARCHIVOS = []                              # lista explícita de rutas; si no está vacía, sustituye a la búsqueda
ARCHIVOS = ["md_bgdy3x3_1750_long.xyz"]
EXCLUIR = []                               # nombres de archivo que no se analizan

# --- Tiempo entre frames guardados (fs) = paso de MD × cada cuántos pasos se escribe un frame.
#     pw.x da dt en unidades atómicas de Rydberg: 1 a.u. = 0.048378 fs (dt = 40 → 1.935 fs).
#     MD de ASE: timestep × loginterval.
DT_FRAME_FS = 1.0                         # obligatorio salvo que la línea de comentario traiga time_fs / time_ps / time
DT_FRAME_POR_ARCHIVO = {}                  # {"md_bgdy3x3_1750_long.xyz": 96.76} si cambia entre archivos

# --- Temperatura nominal: se lee del nombre del archivo o de sus carpetas (1750, 1750K, 2250K_5ps…)
T_POR_ARCHIVO = {}                         # {"nombre.xyz": 1750} si no se puede leer del nombre
T_EXCLUIR = []

DEMO = False                               # True: genera trayectorias sintéticas 3x3 y 6x6 en .xyz y las analiza

# --- Tratamiento de las trayectorias
DESCARTE_INICIAL_PS = 0.5                  # equilibrado inicial que no entra en las tasas
T_REF_FS = 100.0                           # los enlaces de referencia se toman en los frames de este primer intervalo (al menos 1)

# --- Enlaces y criterio de rotura (Å, fs)
PAR_ENLACE = ("B", "C")
R_ENLACE_REF = 1.9                         # enlace = par a menos de esta distancia al inicio (lámina perfecta)
R_ROTURA = 2.2                             # roto si d > R_ROTURA ...
R_UNION = 1.8                              # ... y vuelve a intacto si d < R_UNION
T_PERSIST_FS = 50.0                        # durante al menos este tiempo seguido ...
MIN_FRAMES_PERSIST = 2                     # ... y al menos estos frames (con frames espaciados manda este)

# --- Clasificación de cada rotura
R_VECINO = 1.85                            # Å: vecinos de un átomo (cualquier especie) para contar su coordinación
TIPO = "libre"                             # tipo que entra en Arrhenius y en la nucleación: "libre", "reorganizacion" o "todas"

# --- Estimador para Arrhenius: "primera" (nucleación: primera rotura del TIPO) o "todas" (todas las del TIPO)
ESTIMADOR = "primera"

# --- Propagación y Avrami
R_PROPAGACION = 8.0                        # Å entre puntos medios de enlaces
DT_PROPAGACION_PS = 1.0                    # ventana tras la primera rotura
X_MAX_AVRAMI = 0.3                         # ajuste de Avrami solo en el tramo inicial
MIN_ROTURAS_AVRAMI = 10                    # con menos roturas X(t) son unos pocos escalones y el ajuste no tiene sentido

# --- Lámina (extrapolación)
A_PRIM = 11.847                            # Å, celda primitiva C12B2
ENLACES_POR_CELDA = 6                      # enlaces B–C por celda primitiva

SALIDA = Path("resultados_rotura_BC")

## 2. Librerías y estilo de las figuras

In [ ]:
import re, warnings, itertools
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import chi2, beta
from scipy.optimize import minimize

KB_EV = 8.617333262e-5      # eV/K

INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e5e4e0", "#fcfcfb"
SERIE = ["#2a78d6", "#eb6834", "#1baf7a"]          # un color por tamaño de celda, en orden fijo
COLOR_TIPO = {"libre": SERIE[1], "reorganizacion": SERIE[0]}
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.markersize": 8, "font.size": 10, "legend.frameon": False,
})
pd.set_option("display.width", 180)

## 3. Lectura de trayectorias

Cada archivo se lee entero con `ase.io.read(archivo, index=":")`. De cada frame se guardan la celda (sirve también para NPT), las posiciones y, si el archivo trae momentos, la temperatura instantánea, que se compara con la nominal.

**Tiempo.** Si la línea de comentario trae `time_fs`, `time_ps` o `time` (en fs), se usa. Si no, $t = \text{frame} \times$ `DT_FRAME_FS`.

**Periodicidad.** Se respeta `pbc` del archivo: con `pbc="T T F"` no se aplica imagen mínima en z, así que un átomo que sale del plano no aparece cerca de otro a través del vacío. Las posiciones pueden estar sin envolver.

In [ ]:
CLAVES_TIEMPO = {"time_fs": 1.0, "time_ps": 1000.0, "time": 1.0}     # clave de la línea de comentario → factor a fs
RE_T = re.compile(r"(?<![\dx.])(\d{3,4})(?=[Kk_\-.]|$)")


def lee_xyz(archivo):
    from ase.io import read
    frames = read(archivo, index=":")
    a0 = frames[0]
    cells = np.array([a.cell.array for a in frames])
    if np.abs(cells[:, :2]).sum() == 0:
        raise ValueError(f"{archivo}: sin celda; la línea de comentario debe llevar Lattice=\"...\"")
    for c in cells:                                    # celdas 2D sin vector z: se añade vacío
        if np.linalg.norm(c[2]) < 1e-6:
            c[2] = [0.0, 0.0, 100.0]
    pbc = np.array(a0.pbc, bool)
    if not pbc[:2].all():
        warnings.warn(f"{archivo}: pbc = {pbc}; se usa T T F")
        pbc = np.array([True, True, False])
    t_fs = None
    for clave, factor in CLAVES_TIEMPO.items():
        if all(clave in a.info for a in frames):
            t_fs = np.array([float(a.info[clave]) for a in frames]) * factor
            break
    T_mom = np.array([a.get_temperature() if "momenta" in a.arrays else np.nan for a in frames])
    return dict(simbolos=a0.get_chemical_symbols(), cells=cells, pbc=pbc,
                pos=np.array([a.positions for a in frames], dtype=np.float32), t_fs=t_fs, T_mom=T_mom)


def T_del_nombre(ruta):
    for parte in [Path(ruta).stem] + [p.name for p in Path(ruta).parents]:
        m = RE_T.findall(parte)
        if m:
            return float(m[0])
    return np.nan


def tam_celda(simbolos):
    nB = sum(s == "B" for s in simbolos)
    m = int(round(np.sqrt(nB / 2)))
    return f"{m}x{m}" if 2 * m * m == nB else f"{len(simbolos)} át"

## 4. Detección de roturas, clasificación y estimadores

- **Imagen mínima exacta**: en una celda hexagonal (60°) redondear las coordenadas fraccionarias no siempre da la imagen más cercana; se prueban también las imágenes vecinas en las direcciones periódicas.
- **Enlaces**: pares B–C cuya distancia mediana en los frames de los primeros `T_REF_FS` es menor que `R_ENLACE_REF`. En BGDY debe haber 3 por átomo de B; si no, se avisa.
- **Histéresis**: un enlace pasa a *roto* si supera `R_ROTURA` durante al menos `T_PERSIST_FS` (y al menos `MIN_FRAMES_PERSIST` frames) seguidos, y vuelve a *intacto* si baja de `R_UNION` el mismo tiempo. El suceso se fecha al inicio de esa ventana; los estiramientos térmicos transitorios no cuentan.
- **Clasificación**: se cuentan los vecinos (a menos de `R_VECINO`, de cualquier especie) del B y del C durante la ventana de persistencia y se comparan con los que tenían en la lámina perfecta (primer frame: 3 el B, 2 el C). Si alguno de los dos se queda por debajo, la rotura es **libre**: queda un enlace colgante. Si los dos mantienen al menos esa coordinación gracias a enlaces nuevos, es una **reorganización**. Los enlaces formados en la ventana se listan en la tabla de sucesos (`enlaces_formados`).
- **Exposición**: tiempo-enlace en riesgo, la suma frame a frame de enlaces intactos × Δt.
- **Intervalos**: Poisson exacto (Garwood); con 0 roturas, la tasa es 0 y el intervalo da una cota superior.

In [ ]:
def _imagenes(pbc):
    rangos = [(-1, 0, 1) if p else (0,) for p in pbc]
    return np.array(list(itertools.product(*rangos)), float)


def mic(v, cell, pbc):
    # vector de imagen mínima de v (..., 3) en una celda fija
    pbc = np.asarray(pbc, bool)
    f = v @ np.linalg.inv(cell)
    f[..., pbc] -= np.round(f[..., pbc])
    cand = (f[..., None, :] + _imagenes(pbc)) @ cell
    k = np.argmin((cand ** 2).sum(-1), axis=-1)
    return np.take_along_axis(cand, k[..., None, None], axis=-2)[..., 0, :]


def distancias(pos, cells, pares, pbc, bloque=200):
    # distancia de cada par en cada frame, con la celda de cada frame
    pbc = np.asarray(pbc, bool); S = _imagenes(pbc); out = []
    for a in range(0, len(pos), bloque):
        p, c = pos[a:a + bloque].astype(float), cells[a:a + bloque]
        f = np.einsum("fbi,fij->fbj", p[:, pares[:, 1]] - p[:, pares[:, 0]], np.linalg.inv(c))
        f[..., pbc] -= np.round(f[..., pbc])
        cand = np.einsum("fbsi,fij->fbsj", f[:, :, None, :] + S, c)
        out.append(np.sqrt((cand ** 2).sum(-1).min(-1)))
    return np.vstack(out)


def enlaces(simbolos, pos_ref, cells_ref, pbc, par=PAR_ENLACE, rmax=R_ENLACE_REF):
    # enlace = par con distancia mediana < rmax en los frames de referencia (la mediana evita perder
    # un enlace por una fluctuación térmica en un solo frame)
    s = np.array(simbolos)
    pos0, cell = pos_ref[0].astype(float), cells_ref[0]
    ia, ib = np.where(s == par[0])[0], np.where(s == par[1])[0]
    cand = []
    for i in ia:
        d = np.linalg.norm(mic(pos0[ib] - pos0[i], cell, pbc), axis=1)
        cand += [(i, j) for j in ib[d < rmax + 0.6] if par[0] != par[1] or j > i]
    cand = np.array(cand, dtype=int)
    pares = cand[np.median(distancias(pos_ref, cells_ref, cand, pbc), axis=0) < rmax]
    medios = pos0[pares[:, 0]] + 0.5 * mic(pos0[pares[:, 1]] - pos0[pares[:, 0]], cell, pbc)
    return pares, medios


def n_persist(dt_fs, t_pers=T_PERSIST_FS, min_frames=MIN_FRAMES_PERSIST):
    return max(min_frames, int(np.ceil(t_pers / dt_fs)))


def detecta(D, npers, r_rot=R_ROTURA, r_uni=R_UNION):
    nf, nb = D.shape
    estado = np.zeros(nb, bool)
    racha = np.zeros(nb, int)
    estados = np.zeros((nf, nb), bool)
    roturas, uniones = [], []
    for t in range(nf):
        cond = np.where(estado, D[t] < r_uni, D[t] > r_rot)
        racha = np.where(cond, racha + 1, 0)
        for b in np.where(racha >= npers)[0]:
            t0 = t - npers + 1
            (uniones if estado[b] else roturas).append((t0, int(b)))
            estado[b] = ~estado[b]
            estados[t0:t + 1, b] = estado[b]
            racha[b] = 0
        estados[t] = estado
    return dict(roturas=roturas, uniones=uniones, estados=estados, npers=npers)


def vecinos(pos_f, cell, pbc, atomos, r=R_VECINO):
    atomos = np.asarray(atomos)
    pos_f = pos_f.astype(float)
    d = np.linalg.norm(mic(pos_f[None] - pos_f[atomos, None], cell, pbc), axis=-1)
    d[np.arange(len(atomos)), atomos] = np.inf
    return [set(np.where(fila < r)[0].tolist()) for fila in d]


def clasifica(roturas, pares, simbolos, pos, cells, pbc, npers, r=R_VECINO):
    # "libre" si el B o el C quedan con menos vecinos que en la lámina perfecta (frame 0: B 3, C 2), es decir,
    # con un enlace colgante; "reorganizacion" si los dos conservan esa coordinación gracias a enlaces nuevos.
    # Vecinos tras la rotura: los presentes en al menos la mitad de los frames de la ventana de persistencia.
    tipos, nuevos = [], []
    for t0, b in roturas:
        i, j = (int(x) for x in pares[b])
        ref = vecinos(pos[0], cells[0], pbc, [i, j], r)
        ref[0].add(j); ref[1].add(i)
        ta = max(t0 - 1, 0)
        antes = vecinos(pos[ta], cells[ta], pbc, [i, j], r)
        F = range(t0, min(t0 + npers, len(pos)))
        cuenta = [Counter(), Counter()]
        for f in F:
            for k, s in enumerate(vecinos(pos[f], cells[f], pbc, [i, j], r)):
                cuenta[k].update(s - {i, j})
        despues = [{a for a, c in ck.items() if c >= len(F) / 2} for ck in cuenta]
        libre = len(despues[0]) < len(ref[0]) or len(despues[1]) < len(ref[1])
        tipos.append("libre" if libre else "reorganizacion")
        nuevos.append(", ".join(f"{simbolos[x]}{x}–{simbolos[y]}{y}"
                                for x, d, a in ((i, despues[0], antes[0]), (j, despues[1], antes[1])) for y in sorted(d - a)))
    return tipos, nuevos


def ventana(det, tipos, w0, w1, dt_fs, tipo=TIPO):
    # Sucesos y exposiciones dentro de la ventana de análisis [w0, w1) (tras el descarte inicial).
    # Los enlaces ya rotos al empezar la ventana no están en riesgo. Si alguno se rompió durante el
    # descarte, la trayectoria no entra en el estimador de nucleación (su primera rotura no fue a la T de trabajo).
    est = det["estados"]; dt_ps = dt_fs / 1000.0
    dentro = [k for k, (t, _) in enumerate(det["roturas"]) if w0 <= t < w1]
    rot = [det["roturas"][k] for k in dentro]
    tip = [tipos[k] for k in dentro]
    uni = [(t, b) for t, b in det["uniones"] if w0 <= t < w1]
    rotos_ini = int(est[w0].sum()) if w0 < len(est) else 0
    sel = [t for (t, _), ti in zip(rot, tip) if tipo == "todas" or ti == tipo]
    if rotos_ini == 0:
        fin = sel[0] if sel else w1
        n1, expo1 = int(bool(sel)), (~est[w0:fin]).sum() * dt_ps
    else:
        n1, expo1 = 0, 0.0
    return dict(roturas=rot, tipos=tip, uniones=uni, rotos_al_inicio=rotos_ini,
                n_libre=tip.count("libre"), n_reorg=tip.count("reorganizacion"),
                expo_intacto=(~est[w0:w1]).sum() * dt_ps, expo_roto=est[w0:w1].sum() * dt_ps,
                n_primera=n1, expo_primera=expo1)


def tasa_poisson(n, expo, nivel=0.95):
    a = 1.0 - nivel
    if expo <= 0:
        return np.nan, (np.nan, np.nan)
    lo = 0.0 if n == 0 else chi2.ppf(a / 2, 2 * n) / 2
    hi = chi2.ppf(1 - a / 2, 2 * n + 2) / 2
    return n / expo, (lo / expo, hi / expo)


def arrhenius(T, n, expo):
    # Máxima verosimilitud de Poisson para k = A exp(-Ea/kB T); necesita roturas en >= 2 temperaturas
    T, n, expo = (np.asarray(x, float) for x in (T, n, expo))
    ok = expo > 0
    T, n, expo = T[ok], n[ok], expo[ok]
    if (n > 0).sum() < 2:
        return None
    x = 1.0 / (KB_EV * T)
    nll = lambda p: np.sum(expo * np.exp(p[0] - p[1] * x) - n * (p[0] - p[1] * x))
    m = n > 0
    pend, orde = np.polyfit(x[m], np.log(n[m] / expo[m]), 1)
    res = minimize(nll, [orde, -pend], method="BFGS")
    err = np.sqrt(np.diag(res.hess_inv))
    return dict(Ea_eV=res.x[1], err_Ea=err[1], lnA=res.x[0], err_lnA=err[0], A_ps=np.exp(res.x[0]))


def vida_lamina(k_ps, area_nm2):
    dens = ENLACES_POR_CELDA / (np.sqrt(3) / 2 * (A_PRIM / 10) ** 2)   # enlaces B–C por nm²
    return 1.0 / (k_ps * dens * area_nm2)

## 5. Modo demo (solo si `DEMO = True`)

Genera trayectorias .xyz como las tuyas (extxyz, `pbc="T T F"`, sin tiempo en el archivo): BGDY 3x3 (3 réplicas por T) y 6x6 (2 réplicas por T), un frame cada 10 fs, unos 80 MB en `./demo_xyz_bgdy`. Las roturas siguen una tasa conocida por enlace ($E_a$ = 3 eV, $k$(2500 K) = 3·10⁻³ ps⁻¹) y, tras cada rotura, los enlaces a menos de 8 Å se rompen 20 veces más rápido (cascada). Todas las roturas sintéticas son de tipo libre: el C se separa del plano. Sirve para comprobar el cuaderno de principio a fin; no son datos físicos. Con tan pocas trayectorias, $E_a$ sale con un error de 1–2 eV.

In [ ]:
def bgdy(m, a=A_PRIM):
    cell = np.array([[a, 0, 0], [a / 2, a * np.sqrt(3) / 2, 0], [0, 0, 20.0]])
    B = np.array([[1 / 3, 1 / 3, 0.5], [2 / 3, 2 / 3, 0.5]]) @ cell
    seg = np.array([1.50, 1.23, 1.35, 1.23, 1.50])          # B-C, C#C, C-C, C#C, C-B
    pos, sym = [B[0], B[1]], ["B", "B"]
    for s in ([0, 0, 0], [-1, 0, 0], [0, -1, 0]):
        v = B[1] + np.array(s) @ cell - B[0]
        for f in np.cumsum(seg)[:-1] / seg.sum():
            pos.append(B[0] + f * v); sym.append("C")
    pos = np.array(pos)
    P = np.vstack([pos + i * cell[0] + j * cell[1] for i in range(m) for j in range(m)])
    C = cell.copy(); C[0] *= m; C[1] *= m
    return sym * (m * m), P, C


def simula(m, T, dur_ps, dt_fs, cada, rng, Ea=3.0, k2500=3e-3, k_union=0.3, cascada=20.0, r_casc=8.0):
    sym, ref, cell = bgdy(m)
    pbc = (True, True, False)
    pares, medios = enlaces(sym, ref[None], cell[None], pbc)
    dm = np.linalg.norm(mic(medios[:, None] - medios[None], cell, pbc), axis=2)
    vecinos_casc = (dm < r_casc) & (dm > 0)
    k = k2500 * np.exp(-Ea / KB_EV * (1 / T - 1 / 2500))
    sig = 0.05 * np.sqrt(T / 1000)
    n = int(dur_ps * 1000 / dt_fs)
    roto = np.zeros(len(pares), bool)
    salida = []
    for t in range(n):
        kk = k * np.where(vecinos_casc[:, roto].any(axis=1), cascada, 1.0)
        u = rng.random(len(pares))
        roto = np.where(roto, u > k_union * dt_fs / 1000, u < kk * dt_fs / 1000)
        if t % cada == 0:
            p = ref + rng.normal(0, sig, ref.shape)
            for b in np.where(roto)[0]:
                p[pares[b][1], 2] += 2.2                      # el C del enlace roto sale del plano
            salida.append(p + np.array([0.3, -0.2, 0]) * t / n)      # deriva: posiciones sin envolver
    return sym, cell, np.array(salida)


if DEMO:
    from ase import Atoms
    from ase.io import write
    rng = np.random.default_rng(11)
    CARPETA_XYZ, ARCHIVOS, PATRON_XYZ = Path("demo_xyz_bgdy"), [], "*.xyz"
    DT_FRAME_FS, DT_FRAME_POR_ARCHIVO, T_POR_ARCHIVO = 10.0, {}, {}
    DESCARTE_INICIAL_PS = 0.2                                       # los datos sintéticos no tienen fase de calentamiento
    CARPETA_XYZ.mkdir(exist_ok=True)
    for T in (2250, 2500, 2750):
        for m, reps, dur in ((3, "abc", 3.0), (6, "ab", 4.0)):
            for c in reps:
                sym, cell, pos = simula(m, T, dur, 1.0, 10, rng)
                write(CARPETA_XYZ / f"md_bgdy{m}x{m}_{T}_{c}.xyz",
                      [Atoms(sym, positions=p, cell=cell, pbc=[True, True, False]) for p in pos], format="extxyz")
    print("dataset sintético en", CARPETA_XYZ.resolve(), "| verdad: Ea = 3.0 eV, k(2500 K) = 3e-3 ps⁻¹ por enlace, cascada ×20")

In [ ]:
rutas = [Path(a) for a in ARCHIVOS] if ARCHIVOS else sorted(Path(CARPETA_XYZ).rglob(PATRON_XYZ))
rutas = [p for p in rutas if p.name not in EXCLUIR]
assert rutas, f"No hay trayectorias: revisa CARPETA_XYZ / PATRON_XYZ / ARCHIVOS (o pon DEMO = True)"
stems = Counter(p.stem for p in rutas)
filas = []
for p in rutas:
    filas.append(dict(id=p.stem if stems[p.stem] == 1 else f"{p.parent.name}/{p.stem}", archivo=p.name, ruta=str(p),
                      T_K=float(T_POR_ARCHIVO.get(p.name, T_del_nombre(p))),
                      dt_frame_fs=DT_FRAME_POR_ARCHIVO.get(p.name, DT_FRAME_FS)))
inventario = pd.DataFrame(filas)
inventario = inventario[~inventario.T_K.isin(T_EXCLUIR)].reset_index(drop=True)
if inventario.T_K.isna().any():
    warnings.warn("Hay archivos sin temperatura en el nombre: complétalos en T_POR_ARCHIVO "
                  "(si no, se usará la de los momentos redondeada a 50 K)")
if inventario.dt_frame_fs.isna().any():
    warnings.warn("Hay archivos sin DT_FRAME_FS: solo se podrán leer si la línea de comentario trae el tiempo")
inventario[["id", "T_K", "dt_frame_fs", "ruta"]]

## 6. Inventario de trayectorias

Una fila por archivo, con la temperatura leída del nombre y el tiempo entre frames. **Revisa esta tabla antes de seguir.**

## 7. Lectura, descarte inicial, detección y clasificación

La detección se hace sobre la trayectoria completa; los recuentos y exposiciones, solo en la ventana posterior al descarte. `rotos_al_inicio` > 0 indica que la lámina ya se rompió durante el descarte: esa trayectoria no entra en la tasa de nucleación (si pasa a menudo, reduce `DESCARTE_INICIAL_PS`). `T_momentos` es la temperatura media de los momentos en la ventana analizada, para comprobar la nominal.

In [ ]:
resultados, sucesos = {}, []
for _, fila in inventario.iterrows():
    tr = lee_xyz(fila.ruta)
    nf = len(tr["pos"])
    if tr["t_fs"] is not None:
        t = (tr["t_fs"] - tr["t_fs"][0]) / 1000.0
        dt_fs = float(np.median(np.diff(tr["t_fs"])))
    elif not pd.isna(fila.dt_frame_fs):
        dt_fs = float(fila.dt_frame_fs); t = np.arange(nf) * dt_fs / 1000.0
    else:
        raise ValueError(f"[{fila.id}] el archivo no trae el tiempo: fija DT_FRAME_FS (o DT_FRAME_POR_ARCHIVO['{fila.archivo}'])")
    npers = n_persist(dt_fs)
    nref = max(1, int(np.searchsorted(t, t[0] + T_REF_FS / 1000.0)))
    pares, medios = enlaces(tr["simbolos"], tr["pos"][:nref], tr["cells"][:nref], tr["pbc"])
    nB = tr["simbolos"].count("B")
    if PAR_ENLACE == ("B", "C") and len(pares) != 3 * nB:
        print(f"[{fila.id}] aviso: {len(pares)} enlaces B–C detectados, se esperaban {3 * nB}")
    w0 = int(np.searchsorted(t, t[0] + DESCARTE_INICIAL_PS)); w1 = nf
    T_mom = float(np.nanmean(tr["T_mom"][w0:w1])) if np.isfinite(tr["T_mom"][w0:w1]).any() else np.nan
    if pd.isna(fila.T_K):
        fila = fila.copy(); fila["T_K"] = float(np.round(T_mom / 50) * 50)
        print(f"[{fila.id}] T del nombre no encontrada: se usa {fila.T_K:.0f} K (momentos: {T_mom:.0f} K)")
    elif np.isfinite(T_mom) and abs(T_mom - fila.T_K) > 0.1 * fila.T_K:
        print(f"[{fila.id}] aviso: T de los momentos {T_mom:.0f} K frente a {fila.T_K:.0f} K del nombre")
    D = distancias(tr["pos"], tr["cells"], pares, tr["pbc"])     # trayectoria completa: la detección necesita la historia
    det = detecta(D, npers)
    tipos, nuevos = clasifica(det["roturas"], pares, tr["simbolos"], tr["pos"], tr["cells"], tr["pbc"], npers)
    resultados[fila.id] = dict(fila=fila, tam=tam_celda(tr["simbolos"]), t=t, w0=w0, w1=w1, dt_fs=dt_fs, npers=npers,
                               simbolos=tr["simbolos"], pos=tr["pos"], cells=tr["cells"], pbc=tr["pbc"], T_mom=T_mom,
                               cell=tr["cells"][0], pares=pares, medios=medios, D=D, det=det, tipos=tipos,
                               ven=ventana(det, tipos, w0, w1, dt_fs))
    s = tr["simbolos"]
    for (t0, b), tipo, nv in zip(det["roturas"], tipos, nuevos):
        i, j = pares[b]
        reunion = [tu for tu, bu in det["uniones"] if bu == b and tu > t0]
        sucesos.append(dict(id=fila.id, tam=resultados[fila.id]["tam"], T_K=fila.T_K, frame=t0, t_ps=round(t[t0], 3),
                            enlace=f"{s[i]}{i}–{s[j]}{j}", tipo=tipo, enlaces_formados=nv, en_descarte=t0 < w0,
                            se_reune_en_ps=round(t[reunion[0]] - t[t0], 3) if reunion else None))
TAMS = sorted({r["tam"] for r in resultados.values()}, key=lambda s: (len(s), s))
COLOR_TAM = {t: SERIE[i] if i < len(SERIE) else INK2 for i, t in enumerate(TAMS)}
dts = sorted({r["dt_fs"] for r in resultados.values()})
if max(dts) > T_PERSIST_FS:
    print(f"Frames cada {max(dts):g} fs: la persistencia efectiva es {MIN_FRAMES_PERSIST} frames "
          f"({MIN_FRAMES_PERSIST * max(dts):g} fs). Roturas que se cierran antes no se ven.")

def resumen(r):
    ven, t = r["ven"], r["t"]
    rot = ven["roturas"]
    prim_lib = [t0 for (t0, _), ti in zip(rot, ven["tipos"]) if ti == "libre"]
    return dict(id=r["fila"].id, tam=r["tam"], T_K=r["fila"].T_K, T_momentos=round(r["T_mom"], 0),
                n_frames=len(t), dt_frame_fs=round(r["dt_fs"], 4), analizado_ps=round((r["w1"] - r["w0"]) * r["dt_fs"] / 1000, 3),
                n_enlaces=len(r["pares"]), rotos_al_inicio=ven["rotos_al_inicio"],
                n_roturas=len(rot), n_libres=ven["n_libre"], n_reorg=ven["n_reorg"], n_uniones=len(ven["uniones"]),
                t_primera_ps=round(t[rot[0][0]], 3) if rot else None,
                t_primera_libre_ps=round(t[prim_lib[0]], 3) if prim_lib else None, d_max_A=round(float(r["D"].max()), 2))

tabla_tray = pd.DataFrame([resumen(r) for r in resultados.values()]).sort_values(["T_K", "tam", "id"]).reset_index(drop=True)
tabla_sucesos = pd.DataFrame(sucesos, columns=["id", "tam", "T_K", "frame", "t_ps", "enlace", "tipo", "enlaces_formados",
                                               "en_descarte", "se_reune_en_ps"])
display(tabla_tray)
print(f"{len(tabla_sucesos)} roturas en total (primeras 40):")
tabla_sucesos.head(40)

## 8. Comprobación visual de las roturas

Primero, todas las trayectorias: cada rotura en su instante, por tipo, y las reuniones. Después, la distancia de los enlaces que se rompen en las primeras trayectorias con roturas (hasta 6, tres enlaces cada una), con los dos umbrales. La banda gris es el descarte inicial. Las líneas discontinuas son los umbrales de rotura (`R_ROTURA`) y de reunión (`R_UNION`); la escala se corta en 8 Å. Una rotura real queda por encima de `R_ROTURA` de forma sostenida; un pico aislado no debe contar.

In [ ]:
orden = tabla_tray.id.tolist()[:40]
fig, ax = plt.subplots(figsize=(8.5, 0.34 * len(orden) + 1.4))
for y, id_ in enumerate(orden):
    r = resultados[id_]; t = r["t"]
    ax.plot([t[0], t[-1]], [y, y], color=GRID, lw=5, solid_capstyle="butt", zorder=1)
    ax.plot([t[0], t[min(r["w0"], len(t) - 1)]], [y, y], color=INK2, lw=5, alpha=0.25, solid_capstyle="butt", zorder=1)
    for (t0, _), tipo in zip(r["det"]["roturas"], r["tipos"]):
        lib = tipo == "libre"
        ax.plot(t[t0], y, "o", ms=8, mew=1.5, color=COLOR_TIPO[tipo], mfc=COLOR_TIPO[tipo] if lib else SURF, zorder=3)
    for t0, _ in r["det"]["uniones"]:
        ax.plot(t[t0], y, "|", ms=10, mew=1.5, color=INK2, zorder=2)
ax.set_yticks(range(len(orden)), [f"{i}  ({resultados[i]['fila'].T_K:.0f} K)" for i in orden], fontsize=8)
ax.invert_yaxis(); ax.grid(axis="y", visible=False); ax.set_xlabel("t (ps)")
ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, color=COLOR_TIPO["libre"], label="rotura libre"),
                   Line2D([], [], ls="", marker="o", ms=8, mew=1.5, color=COLOR_TIPO["reorganizacion"], mfc=SURF, label="reorganización"),
                   Line2D([], [], ls="", marker="|", ms=10, mew=1.5, color=INK2, label="reunión")],
          loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8, borderaxespad=-2.2)
ax.set_title("Roturas B–C por trayectoria", loc="left", fontsize=11, pad=26)
fig.tight_layout(); plt.show()

con_roturas = [r for r in resultados.values() if r["det"]["roturas"]]
if not con_roturas:
    print("Ninguna trayectoria tiene roturas con estos umbrales.")
else:
    muestra = con_roturas[:6]
    n = len(muestra); ncol = min(3, n); nfil = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nfil, ncol, figsize=(4.8 * ncol, 3.0 * nfil), squeeze=False, sharey=True)
    for ax, r in zip(axs.flat, muestra):
        t = r["t"]
        if r["w0"] > 0:
            ax.axvspan(t[0], t[min(r["w0"], len(t) - 1)], color=GRID, alpha=0.6, lw=0)
        rotos = list(dict.fromkeys(b for _, b in r["det"]["roturas"]))[:3]
        tipo_de = {}
        for (_, b), ti in zip(r["det"]["roturas"], r["tipos"]):
            tipo_de.setdefault(b, ti)
        s = r["simbolos"]
        for c, b in enumerate(rotos):
            i, j = r["pares"][b]
            ax.plot(t, r["D"][:, b], color=SERIE[c], lw=1.5,
                    label=f"{s[i]}{i}–{s[j]}{j} ({'libre' if tipo_de[b] == 'libre' else 'reorg.'})")
        for y in (R_ROTURA, R_UNION):
            ax.axhline(y, color=INK2, lw=1, ls="--", zorder=0)
            ax.text(1.01, y, f"{y:g}", transform=ax.get_yaxis_transform(), fontsize=7.5, color=INK2, ha="left", va="center")
        ymax = max(3.9, min(r["D"][:, rotos].max() + 0.7, 20.0))
        ax.set_ylim(1.0, ymax)
        ax.set_title(f"{r['fila'].id}  ({r['tam']}, {r['fila'].T_K:.0f} K)", fontsize=9, loc="left")
        ax.set_xlabel("t (ps)")
        ax.legend(fontsize=7.5, loc="upper left", ncol=1 if len(rotos) > 1 else 1, handlelength=1.2)
    for ax in axs[:, 0]:
        ax.set_ylabel("distancia B–C (Å)")
    for ax in list(axs.flat)[n:]:
        ax.set_visible(False)
    fig.tight_layout(); plt.show()

## 9. Tasa de rotura por enlace, por temperatura y tamaño

Se suman sucesos y exposiciones de todas las trayectorias con la misma T y el mismo tamaño de celda.
- `k_todas`, `k_libre`, `k_reorg`: todas las roturas, o las de cada tipo, sobre el tiempo-enlace intacto. Son riesgos en competencia: `k_todas = k_libre + k_reorg`.
- `k_primera`: solo la primera rotura del `TIPO` elegido en cada trayectoria, con exposición hasta ese momento. Es la **nucleación** y no la contaminan las cascadas. Si sale claramente menor que `k_<TIPO>`, hay propagación.
- `k_union`: reformación de enlaces rotos.

**La comprobación clave**: a la misma T, la tasa debe coincidir entre tamaños (los intervalos deben solaparse).

In [ ]:
filas = []
for (T, tam), grupo in tabla_tray.groupby(["T_K", "tam"]):
    vs = [resultados[i]["ven"] for i in grupo.id]
    e_i = sum(v["expo_intacto"] for v in vs)
    n_r, n_l, n_o = (sum(len(v["roturas"]) for v in vs), sum(v["n_libre"] for v in vs), sum(v["n_reorg"] for v in vs))
    n_1, e_1 = sum(v["n_primera"] for v in vs), sum(v["expo_primera"] for v in vs)
    n_u, e_r = sum(len(v["uniones"]) for v in vs), sum(v["expo_roto"] for v in vs)
    n_tipo = {"todas": n_r, "libre": n_l, "reorganizacion": n_o}[TIPO]
    k1, (lo1, hi1) = tasa_poisson(n_1, e_1)
    kt, (lot, hit) = tasa_poisson(n_tipo, e_i)
    filas.append(dict(T_K=T, tam=tam, n_tray=len(vs), n_excluidas_nucl=sum(v["rotos_al_inicio"] > 0 for v in vs),
                      n_roturas=n_r, n_libres=n_l, n_reorg=n_o, expo_enlace_ps=e_i,
                      k_todas=tasa_poisson(n_r, e_i)[0], k_libre=tasa_poisson(n_l, e_i)[0], k_reorg=tasa_poisson(n_o, e_i)[0],
                      n_tipo=n_tipo, k_tipo=kt, IC95_lo=lot, IC95_hi=hit,
                      n_primeras=n_1, expo_hasta_primera=e_1, k_primera=k1, IC95_lo_1=lo1, IC95_hi_1=hi1,
                      n_uniones=n_u, k_union=tasa_poisson(n_u, e_r)[0]))
tasas = pd.DataFrame(filas)
print(f"TIPO = {TIPO!r}: k_tipo y k_primera se refieren a ese tipo")
enteros = ("T_K", "tam", "n_tray", "n_excluidas_nucl", "n_roturas", "n_libres", "n_reorg", "n_tipo", "n_primeras", "n_uniones")
tasas.style.format({c: "{:.3g}" for c in tasas.columns if c not in enteros})

## 11. Propagación: ¿el defecto se queda local?

Para cada trayectoria con roturas, a partir de la primera (de cualquier tipo):
- `roturas_cerca`: otras roturas a menos de `R_PROPAGACION` del primer enlace roto en los `DT_PROPAGACION_PS` siguientes; `roturas_lejos`, las de más lejos.
- `frac_intactos_lejos_min`: fracción mínima de enlaces intactos entre los que están lejos del primer defecto. Si es 1, el resto de la lámina no se ha tocado.

`P_cascada` es la fracción de primeras roturas seguidas de otra cercana, con intervalo de Clopper–Pearson. Si es ≈ 0, el defecto es local y $k$ describe por sí sola la estabilidad.

In [ ]:
filas = []
for r in con_roturas:
    det, t = r["det"], r["t"]
    t0, b0 = det["roturas"][0]
    dist = np.linalg.norm(mic(r["medios"] - r["medios"][b0], r["cell"], r["pbc"]), axis=1)
    lejos = dist > R_PROPAGACION
    tras = [(ti, b) for ti, b in det["roturas"][1:] if t[ti] - t[t0] <= DT_PROPAGACION_PS]
    cerca = [b for _, b in tras if not lejos[b]]
    frac_lejos = (~det["estados"][t0:, lejos]).mean(axis=1).min() if lejos.any() else np.nan
    filas.append(dict(id=r["fila"].id, tam=r["tam"], T_K=r["fila"].T_K, t_primera_ps=round(t[t0], 3),
                      tipo_primera=r["tipos"][0], roturas_cerca=len(cerca), roturas_lejos=len(tras) - len(cerca),
                      frac_intactos_lejos_min=round(frac_lejos, 3)))
propagacion = pd.DataFrame(filas)
if len(propagacion):
    for (tam, T), g in propagacion.groupby(["tam", "T_K"]):
        n, c = len(g), int((g.roturas_cerca > 0).sum())
        lo = beta.ppf(0.025, c, n - c + 1) if c > 0 else 0.0
        hi = beta.ppf(0.975, c + 1, n - c) if c < n else 1.0
        print(f"{tam:>6} {T:.0f} K: P_cascada = {c}/{n} = {c / n:.2f}  (IC 95 %: {lo:.2f}–{hi:.2f})")
propagacion

## 12. Cinética del daño: exponente de Avrami

Fracción de enlaces B–C rotos (de cualquier tipo) $X(t)$ en cada trayectoria, en la representación de Avrami: $\ln[-\ln(1-X)]$ frente a $\ln t$. La pendiente es el exponente $n$ (ajuste solo con $X <$ `X_MAX_AVRAMI`).
- $n \approx 1$: roturas independientes, defectos locales; entonces $\tau \approx 1/k$.
- $n > 1$: el daño nuclea y crece; en 2D, nucleación continua con crecimiento lineal da $n \approx 3$.

$n$ y $\tau$ son intensivos solo si la celda es mayor que el tamaño de una zona dañada: en 3x3 un solo defecto ya ocupa una fracción grande de la celda y $X$ avanza a saltos de 1/54. La comparación útil es entre celdas grandes.

In [ ]:
filas, curvas = [], []
for r in resultados.values():
    X = r["det"]["estados"].mean(axis=1)                    # trayectoria completa, t desde el inicio de la MD
    t = r["t"] - r["t"][0] + r["dt_fs"] / 1000
    m = (X > 0) & (X < X_MAX_AVRAMI)
    n_av = tau = np.nan
    if len(r["det"]["roturas"]) >= MIN_ROTURAS_AVRAMI and m.sum() >= 5:
        n_av, c = np.polyfit(np.log(t[m]), np.log(-np.log(1 - X[m])), 1)
        tau = np.exp(-c / n_av)
        curvas.append((r, t[m], X[m]))
    filas.append(dict(id=r["fila"].id, tam=r["tam"], T_K=r["fila"].T_K, n_roturas_total=len(r["det"]["roturas"]),
                      X_final=round(X[-1], 3), n_avrami=round(n_av, 2), tau_ps=round(tau, 3)))
avrami = pd.DataFrame(filas).sort_values(["T_K", "tam", "id"]).reset_index(drop=True)

if curvas:
    Ts = sorted({r["fila"].T_K for r, _, _ in curvas})
    fig, axs = plt.subplots(1, len(Ts), figsize=(4.2 * len(Ts), 3.6), squeeze=False, sharey=True)
    for ax, T in zip(axs[0], Ts):
        vistos = set()
        for r, tt, XX in curvas:
            if r["fila"].T_K != T:
                continue
            lab = r["tam"] if r["tam"] not in vistos else None; vistos.add(r["tam"])
            ax.plot(np.log(tt), np.log(-np.log(1 - XX)), color=COLOR_TAM[r["tam"]], lw=1.5, label=lab)
        (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
        for n_ref in (1, 3):
            xe = min(x1, x0 + (y1 - y0) / n_ref)
            ax.plot([x0, xe], [y0, y0 + n_ref * (xe - x0)], color=INK2, lw=1, ls="--")
            ax.text(xe - 0.02 * (x1 - x0), y0 + n_ref * (xe - x0) - 0.02 * (y1 - y0), f"n = {n_ref}",
                    fontsize=8, color=INK2, ha="right", va="top", zorder=5,
                    bbox=dict(fc=SURF, ec="none", pad=1.0))
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_title(f"{T:.0f} K", loc="left", fontsize=10); ax.set_xlabel("ln t (ps)")
        ax.legend(title="celda", fontsize=8, title_fontsize=8, loc="lower right")
    axs[0, 0].set_ylabel("ln[−ln(1 − X)]")
    fig.suptitle("Avrami: fracción de enlaces B–C rotos", x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()
else:
    print(f"Ninguna trayectoria tiene {MIN_ROTURAS_AVRAMI} roturas o más: no se ajusta Avrami.")
avrami

## 13. Sensibilidad a los umbrales

Número de roturas del `TIPO` elegido (todas y primeras) por temperatura y tamaño con otros criterios: umbral de rotura y persistencia (en múltiplos de la de trabajo, en frames). Si cambia mucho, el resultado depende del criterio y hay que justificarlo.

In [ ]:
filas = []
for r_rot in (2.0, 2.2, 2.5):
    for fac in (0.5, 1.0, 2.0):
        cuenta = Counter()
        for r in resultados.values():
            npers = max(1, int(round(r["npers"] * fac)))
            d = detecta(r["D"], npers, r_rot=r_rot, r_uni=min(R_UNION, r_rot - 0.2))
            tip, _ = clasifica(d["roturas"], r["pares"], r["simbolos"], r["pos"], r["cells"], r["pbc"], npers)
            v = ventana(d, tip, r["w0"], r["w1"], r["dt_fs"])
            clave = f"{r['fila'].T_K:.0f} K {r['tam']}"
            cuenta[clave + " todas"] += sum(1 for ti in v["tipos"] if TIPO == "todas" or ti == TIPO)
            cuenta[clave + " primeras"] += v["n_primera"]
        filas.append({"R_ROTURA (Å)": r_rot, "persistencia": f"×{fac:g}", **dict(sorted(cuenta.items()))})
sensibilidad = pd.DataFrame(filas)
print(f"TIPO = {TIPO!r}")
sensibilidad

## 14. Extrapolación: vida de láminas de distinto tamaño

Tiempo medio hasta la primera rotura del `TIPO` elegido, $\tau = 1/(k\,\rho\,\text{área})$, con $\rho$ = 6 enlaces B–C por celda primitiva, usando el ajuste de la celda más grande disponible. Fuera del intervalo de temperaturas simulado es una extrapolación de Arrhenius: el intervalo mostrado ($E_a \pm 1\sigma$, a $A$ fijo) crece mucho al bajar T.

In [ ]:
validos = [t for t in TAMS if ajustes.get(t)]
extrapolacion = pd.DataFrame()
if validos:
    aj = ajustes[validos[-1]]
    filas = []
    for T in (300, 600, 1000, 1500, 2000, 2500):
        for etiqueta, area in (("10 nm²", 10.0), ("1 μm²", 1e6), ("1 mm²", 1e12)):
            v = [vida_lamina(np.exp(aj["lnA"] - Ea / (KB_EV * T)), area) * 1e-12
                 for Ea in (aj["Ea_eV"], aj["Ea_eV"] - aj["err_Ea"], aj["Ea_eV"] + aj["err_Ea"])]
            filas.append(dict(T_K=T, area=etiqueta, vida_s=v[0], vida_s_min=v[1], vida_s_max=v[2]))
    extrapolacion = pd.DataFrame(filas)
    print(f"Ajuste usado: celda {validos[-1]}")
    display(extrapolacion.style.format({"vida_s": "{:.2e}", "vida_s_min": "{:.2e}", "vida_s_max": "{:.2e}"}))
else:
    print("Sin ajuste de Arrhenius no se extrapola.")

## 15. Exportar resultados

In [ ]:
SALIDA.mkdir(exist_ok=True)
for nombre, df in (("trayectorias", tabla_tray), ("sucesos", tabla_sucesos), ("tasas_por_T_y_tam", tasas),
                   ("propagacion", propagacion), ("avrami", avrami), ("sensibilidad_umbrales", sensibilidad),
                   ("extrapolacion_vida", extrapolacion)):
    if len(df):
        df.to_csv(SALIDA / f"{nombre}.csv", index=False)
pd.DataFrame({t: a for t, a in ajustes.items() if a}).T.assign(tipo=TIPO, estimador=ESTIMADOR).to_csv(SALIDA / "arrhenius_por_tam.csv")
print("Guardado en", SALIDA.resolve())

## 16. Límites antes de interpretar

- **Frames espaciados**: con un frame cada ~100 fs, una rotura que se cierra antes de `MIN_FRAMES_PERSIST` frames no se ve, y el instante de cada suceso solo se conoce con esa resolución. La tasa de roturas sostenidas no cambia; la de aperturas breves queda subestimada.
- **Clasificación**: depende de `R_VECINO` y de los frames de la ventana de persistencia. Un extremo de cadena colgante que pasa por casualidad cerca de otro átomo puede contarse como reorganización. Revisa la tabla de sucesos (`enlaces_formados`) en los casos dudosos.
- **NVT a área fija**: si la lámina tiende a contraerse o dilatarse al calentarse, la celda fija le impone una tensión. Esa tensión acelera la nucleación y, sobre todo, abre los poros una vez rotos: en una celda fija el daño se propaga como una grieta bajo carga. Compara con MD en NPT en el plano (tensión nula) antes de atribuir la propagación al material.
- **Tamaño**: en 3x3 (3.5 nm) un defecto ya interactúa con sus imágenes periódicas y no caben ondulaciones de longitud de onda larga. La independencia del tamaño se demuestra con al menos dos celdas grandes.
- **Estadística**: con pocas trayectorias habrá pocas roturas libres; las T sin roturas solo dan cotas. Para Arrhenius hacen falta muchas réplicas.
- **Lindemann por pares**: si se usa, que sea en ventana deslizante (no acumulado desde t = 0) y como fracción de pares por encima del umbral, no como media. La media acumulada crece sin límite con cualquier par separado.
- **Espín**: un enlace roto deja un radical y la magnetización cambia. Ni el recuento ni un MLIP lo reflejan, pero importa al comparar energéticas con DFT.